# Caso 10 - Clustering para perfil de consumo energético

El aprendizaje de máquina tiene una amplia gama de aplicaciones para el sector energético. Una muy interesante es la extracción de información sobre el comportamiento del consumo eléctrico. La forma en que un individuo o una familia utiliza la energía a lo largo del día se conoce también como "huella energética".

En este ejercicio, veremos cómo encontrar patrones en los perfiles de carga diarios de un solo hogar usando un algoritmo de agrupación.

El conjunto de datos contiene 2075259 mediciones recogidas entre diciembre de 2006 y noviembre de 2010 (47 meses). Puede encontrarlo [aquí](https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption).


## Resultado Previsto de Aprendizaje
Al finalizar este caso, el estudiante será capaz de:

- Aplicar técnicas de preprocesamiento y escalamiento para preparar datos numéricos para clustering.
- Implementar algoritmos no supervisados de detección de anomalías.
- Implementar algoritmos no supervisados de clústering.

### Contexto del problema
Usted forma parte de un equipo de analistas de una empresa de energía y desea establecer diferentes patrones diarios de consumo de energía en los hogares. Esta información le permitirá en el futuro hacer pronósticos de consumo y también le ayudará a determinar posibles comportamientos anómalos.

# Cargando los datos

Iniciemos cargando las librerías requeridas

In [ ]:
import pandas as pd # Pandas para leer los datos
import matplotlib.pyplot as plt # Matplotlib para visualizaciones
import numpy as np # Numpy para operaciones de tipo algebraico
from sklearn.ensemble import IsolationForest

De toda la información disponible en la base de datos, vamos a usar solamente unas columnas específicas: `'Date', 'Time','Global_active_power'`

In [ ]:
cols_to_use = ['Date', 'Time','Global_active_power']

Ahora sí, cargamos el archivo que contiene los datos:

In [ ]:
df_uci = pd.read_csv('data/household_power_consumption.txt', sep=';', usecols=cols_to_use)

Veamos el resultado

In [ ]:
df_uci

Tenemos entonces una columna de fecha y una columna de hora, ambas numéricas. Como se observa, las medidas se hacen cada minuto. Podemos combinar la fecha y la hora para crear una única variable de tipo `datetime`.

In [ ]:
# creamos la nueva columna con la fecha
df_uci['datetime'] = pd.to_datetime(df_uci['Date'] + ' '+ df_uci['Time'])
# borramos las columnas anteriores
df_uci = df_uci.drop(['Date','Time'], axis=1)
# fijamos la fecha y hora como el index
df_uci = df_uci.set_index('datetime')

df_uci

Veamos qué hay en la columna de consumo:

In [ ]:
df_uci.describe()

Como se observa, la columna `Global Active Power` ha sido procesada como texto, tiene 6534 valores diferentes y la ocurrencia más común es el símbolo '?' con 25979 repeticiones. El símbolo '?' corresponde a lecturas erróneas, y en nuestro contexto debería ser reemplazado por valores `nan`.

In [ ]:
df_uci = df_uci.replace('?', np.nan) # reemplazamos por nan
df_uci.describe()

Ahora analicemos qué ocurre con los datos nulos. Como nuestro análisis será por días, lo primero que haremos será determinar cuál es la distribución de datos nulos por día, esto nos permitirá establecer si hay días con muchos valores faltantes. Para hacerlo, podemos sacar ventaja de que el índice en nuestra base de datos tiene el formato `datetime` y usar la función `resample`

In [ ]:
df_uci.isna().resample('D').sum().hist() # mostramos los nulos por día
plt.ylabel('Conteo')
plt.xlabel('Número de nulos por día')
plt.title('Número de nulos por día')
plt.show()

En un día hay 1440 minutos, como se observa, hay varios días en nuestra base de datos para los que el conteo de nulos cubre casi la totalidad del día. Estos días deben ser retirados de nuestra base de datos, como punto de corte estableceremos 100 puntos faltantes por día.

In [ ]:
# Identificar los días válidos (con menos de 100 nulos)
dias_validos = df_uci.isna().resample('D').sum() < 100

# Filtrar el DataFrame original usando los días válidos
df_uci = df_uci[df_uci.index.normalize().isin(dias_validos[dias_validos['Global_active_power']].index)]

La función `normalize` nos sirve para tomar sólo la fecha del elemento `DateTime`. Veamos ahora cómo se ve la misma gráfica anterior

In [ ]:
df_uci.isna().resample('D').sum().hist() # mostramos los nulos por día
plt.ylabel('Conteo')
plt.xlabel('Número de nulos por día')
plt.title('Número de nulos por día')
plt.show()

Ahora los valores nulos máximos por día son de alrededor de 70. En este punto asumiremos que estos valores nudos corresponden a datos aislados, posiblemente fruto de problemas transitorios de comunicación con el dispositivo, que podemos reemplazar con el último valor real en nuestra base de datos.

In [ ]:
df_uci = df_uci.astype(float).bfill() # reemplazar los valores faltantes por el último dato válido

Ahora nuestra base de datos es numérica, y ya no debe tener datos faltantes

In [ ]:
df_uci.describe()

Para simplificar haremos un promedio sobre los registros de cada hora:

In [ ]:
df_uci_hourly = df_uci.resample('h').mean()
df_uci_hourly

Ahora descompongamos de nuevo la fecha - hora

In [ ]:
df_uci_hourly['hour'] = df_uci_hourly.index.hour
df_uci_hourly

In [ ]:
df_uci_hourly['hour'] = df_uci_hourly.index.hour
df_uci_hourly.index = df_uci_hourly.index.normalize()
df_uci_hourly

Ahora cambiaremos la forma de la base de datos, usando la función `pivot`. La idea es que cada columna sea una hora diferente del día

In [ ]:
df_uci_pivot = df_uci_hourly.pivot(columns='hour', values = 'Global_active_power')
df_uci_pivot

El primer y el último día en nuestra base de datos están incompletos, los podemos sacar de nuestra estadística para quedarnos sólo con días con la información completa

In [ ]:
df_uci_pivot.dropna(inplace = True)
df_uci_pivot

Para entender la gráfica que viene a continuación, veamos qué ocurre cuando transponemos

In [ ]:
df_uci_pivot.T

Grafiquemos entonces el consumo en el primer día

In [ ]:
df_uci_pivot.iloc[0].plot(figsize=(13,8), legend=False, color='blue')

Grafiquemos ahora el consumo de todos los días de la base

In [ ]:
df_uci_pivot.T.plot(figsize=(13,8), legend=False, color='blue', alpha=0.02)

El gráfico anterior muestra todos los perfiles de carga diaria de 1440 días trazados juntos. Podemos ver dos patrones claros de comportamiento del consumo observando las regiones más oscuras (donde se concentran más curvas).

# Detección de anomalías

Intentemos identificar días atípicos, usando el algoritmo de isolation forest, el isolation forest es un algoritmo que nos permite determinar datos atípicos calculando su probabilidad de quedar aislados cuando se ejecuta un árbol de decisión. Como lo que se calcula es una probabilidad de ser atípico, el algoritmo lo único que necesita es una prevalencia esperada de datos atípicos, esta prevalencia se convierte en un número de datos atípicos que el algoritmo escoge ordenando del más probable al menos probable. En nuestro ejercicio, usaremos una prevalencia esperada (contaminación) del 5%.

In [ ]:
# Crear el modelo Isolation Forest
model = IsolationForest(contamination=0.05)

# Ajustar el modelo a los datos
model.fit(df_uci_pivot)

# Obtener las puntuaciones de anomalía
anomaly_scores = model.decision_function(df_uci_pivot)

# Número de outliers detectados
print("Número de días atípicos:", np.count_nonzero(anomaly_scores < 0))
print("Número de días totales:", len(df_uci_pivot))

# Identificar los días atípicos (puntuaciones negativas)
outliers = df_uci_pivot[anomaly_scores < 0]

# Graficar todos los días y los días atípicos
fig, ax = plt.subplots(1,1, figsize = (13,8))
outliers.T.plot(figsize=(13,8), legend=False, color='red', alpha=0.1, ax = ax)
df_uci_pivot.T.plot(figsize=(13,8), legend=False, color='blue', alpha=0.02, ax=ax)


# Añadir leyenda personalizada
from matplotlib.lines import Line2D
custom_lines = [
    Line2D([0], [0], color='blue', alpha=0.5, lw=2, label='Días normales'),
    Line2D([0], [0], color='red', alpha=0.5, lw=2, label='Días anómalos')
]
ax.legend(handles=custom_lines, loc='upper right')

ax.set_title('Perfiles de carga diaria - Días atípicos')
ax.set_xlabel('Hora del día')
ax.set_ylabel('Consumo de energía')


Como se observa, el algoritmo detecta días que, por su configuración, no se ajustan a los patrones usuales de consumo. Creemos entonces una nueva base de datos en la que se excluyen estas anomalías del análisis

In [ ]:
df_uci_clean = df_uci_pivot[anomaly_scores > 0]

# Clustering con K-means

Como ya vimos, K-means es un algoritmo de aprendizaje no supervisado en el que el número de clusters debe definirse a priori. Esto deja la cuestión de cuántos clusters elegir.

### El Coeficiente Silueta (Silhouette Coefficient)

El **Coeficiente de Silueta** es una métrica utilizada para evaluar la calidad de los resultados de un algoritmo de clustering. Mide qué tan bien agrupado está un objeto en comparación con otros clústeres. En esencia, cuantifica qué tan similar es un punto a su propio clúster (cohesión) en comparación con otros clústeres (separación).

El puntaje varía en un rango de **-1 a 1**.

#### ¿Cómo se calcula?

Para cada punto de datos $i$, el coeficiente de silueta $s(i)$ se calcula de la siguiente manera:

1.  **Calcular $a(i)$:** La **distancia media intra-clúster**. Es la distancia promedio del punto $i$ a todos los demás puntos dentro del *mismo* clúster. Un valor pequeño de $a(i)$ indica que el punto está bien asignado a su clúster (alta cohesión).

    $$ a(i) = \frac{1}{|C_k| - 1} \sum_{j \in C_k, i \neq j} d(i, j) $$

    Donde $C_k$ es el clúster al que pertenece el punto $i$, y $d(i, j)$ es la distancia entre los puntos $i$ y $j$.

2.  **Calcular $b(i)$:** La **distancia media al clúster más cercano**. Es la distancia promedio del punto $i$ a todos los puntos en el clúster más cercano (el que no es su propio clúster).

    $$ b(i) = \min_{l \neq k} \left( \frac{1}{|C_l|} \sum_{j \in C_l} d(i, j) \right) $$

    Un valor grande de $b(i)$ significa que el punto está lejos de los otros clústeres (buena separación).

3.  **Calcular el coeficiente de silueta $s(i)$ para el punto $i$**:

    $$ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} $$

El **Coeficiente de Silueta global** para un conjunto de datos completo es simplemente el promedio de $s(i)$ para todos los puntos.

---

#### ¿Cómo interpretar los valores?

* **Puntaje cercano a +1:** Indica que el punto está muy lejos de los clústeres vecinos. Es una buena señal de que el punto está bien asignado a su clúster. 👍
* **Puntaje cercano a 0:** Indica que el punto está muy cerca o en el límite de decisión entre dos clústeres vecinos. 🤔
* **Puntaje cercano a -1:** Indica que el punto probablemente ha sido asignado al clúster incorrecto. 👎

Un puntaje de silueta general más alto indica un modelo de clustering mejor definido y más apropiado.


Vamos a experimentar con un rango de números de cluster (de 2 a 30). Es importante escalar cada periodo dentro del mismo rango para que la magnitud de la carga energética no interfiera en la selección del cluster.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

sillhoute_scores = []
n_cluster_list = np.arange(2,31).astype(int)

X = df_uci_clean.values.copy()

# Very important to scale!
sc = MinMaxScaler()
X = sc.fit_transform(X)

for n_cluster in n_cluster_list:

    kmeans = KMeans(n_clusters=n_cluster)
    cluster_found = kmeans.fit_predict(X)
    sillhoute_scores.append(silhouette_score(X, kmeans.labels_))

Grafiquemos cómo se comporta el Silhouette Score como función del número de clústeres definidos.

In [ ]:
import seaborn as sns
sns.scatterplot(x=np.arange(2,31), y=sillhoute_scores)

Para tomar una decisión, podemos también apoyarnos del método del codo

In [ ]:
inertia = []
for n in n_cluster_list:
    kmeans = KMeans(n_clusters=n)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(n_cluster_list, inertia, marker='o')
plt.xlabel('Número de clústeres')
plt.ylabel('Inercia (Suma de distancias cuadradas)')
plt.title('Método del codo para K-means')
plt.show()

O del método de Calinski-Harabasz

In [ ]:
from sklearn.metrics import calinski_harabasz_score

# calculando el índice Calinski-Harabasz para cada número de clusters
CH_scores = []
for i in range(2, 31):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X)
    score = calinski_harabasz_score(X, kmeans.labels_)
    CH_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 31), CH_scores, marker='o')
plt.title('Índice Calinski-Harabasz para Determinar el Número Óptimo de Clusters')
plt.xlabel('Número de Clusters')
plt.ylabel('Índice Calinski-Harabasz')

Realmente ninguno de los métodos es definitivo, pero los coeficientes de silueta y de Calinski-Harabazs parecen sugerir que con pocos clústeres se obtienen mejores resultados. Vamos entonces a sugerir 3 clústeres.

In [ ]:
# corremos el modelo
kmeans = KMeans(n_clusters=3, random_state=42)
kmeanslabels = kmeans.fit_predict(X)

# creamos una serie con los resultados
kmeanslabels_sr = pd.Series(kmeanslabels, name='cluster_km')

# agregamos la clusterizacion al índice
df_uci_clean = df_uci_clean.set_index(kmeanslabels_sr, append=True )

En este caso, hemos ingresado la información de la clusterización al índice, no como una columna. Esto es necesario para no dañar nuestro mecanismo de graficación de los resultados a partir de la función transpuesta.

In [ ]:
fig, ax= plt.subplots(1,1, figsize=(18,10))
color_list = ['blue','red','green']
cluster_values = sorted(df_uci_clean.index.get_level_values('cluster_km').unique())

for cluster, color in zip(cluster_values, color_list):
    df_uci_clean.xs(cluster, level=1).T.plot(
        ax=ax, legend=False, alpha=0.01, color=color, label= f'Cluster {cluster}'
        )
    df_uci_clean.xs(cluster, level=1).median().plot(
        ax=ax, color=color, alpha=0.9, ls='--'
    )

ax.set_xticks(np.arange(0,24))
ax.set_ylabel('kilowatts')
ax.set_xlabel('hour')

Como podemos ver, K-means encontró tres grupos únicos de perfiles de carga.

El grupo rojo contiene cargas que mantienen un uso constante de energía durante toda la tarde. Quizá se trate de días en los que los ocupantes se quedaron en casa, como los fines de semana y las fechas especiales.

El grupo azul tiene un pico alto por la mañana, un descenso en el uso durante la tarde y alto de nuevo por la noche. Este patrón parece encajar con los días laborables en los que los ocupantes se van al trabajo y/o al colegio.

Por último, el grupo verde muestra días en los que el consumo es bajo durante todo el día. ¿Tal vez se trate de días festivos en los que sólo se dejan encendidos algunos aparatos?

# Clústering Jerárquico

Como no tenemos demasiados datos, podemos intentar usar el algoritmo de clústering jerárquico y analizar los resultados. Este método agrupa los puntos de los más cercanos a los más lejanos y nos permite visualizar la creación de clústeres en un dendrograma

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Calcula la matriz de enlaces (linkage matrix)
Z = linkage(X, method='ward')

# Visualiza el dendrograma
plt.figure(figsize=(16, 6))
dendrogram(Z, truncate_mode='level', p=10, color_threshold=10)
plt.title('Dendrograma de clustering jerárquico (ward)')
plt.xlabel('Muestras')
plt.ylabel('Distancia')
plt.show()

Como se observa, el dendrograma sugiere naturalmente que hay 3 grupos de datos, usemos este resultado para hacer el clústering por aglomeración

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# Definir el número de clusters (puedes cambiar n_clusters según lo que desees analizar)
n_clusters = 3

# Ajustar el modelo de clustering jerárquico aglomerativo
agglo = AgglomerativeClustering(n_clusters=n_clusters)
agglo_labels = agglo.fit_predict(X)

# creamos una serie con los resultados
agglolabels_sr = pd.Series(agglo_labels, name='cluster_aglo')
# Agregar los labels al index del DataFrame
df_uci_clean = df_uci_clean.set_index(agglolabels_sr, append=True )

# Visualizar la cantidad de días por clúster
print(pd.Series(agglo_labels).value_counts())

Los 3 grupos de datos están bien balanceados, revisemos ahora gráficamente los resultados.

In [ ]:
fig, ax= plt.subplots(1,1, figsize=(18,10))
color_list = ['blue','red','green']
cluster_values = sorted(df_uci_clean.index.get_level_values('cluster_aglo').unique())

for cluster, color in zip(cluster_values, color_list):
    df_uci_clean.xs(cluster, level=2).T.plot(
        ax=ax, legend=False, alpha=0.01, color=color, label= f'Cluster {cluster}'
        )
    df_uci_clean.xs(cluster, level=2).median().plot(
        ax=ax, color=color, alpha=0.9, ls='--'
    )

ax.set_xticks(np.arange(0,24))
ax.set_ylabel('kilowatts')
ax.set_xlabel('hour')

Como se observa, los resultados del clústering jerárquico se asemejan mucho a los de KMeans. Tenemos un grupo caracterizado por el consumo constante a lo largo del día. Un grupo caracterizado por dos picos de consumo en la mañana y en la tarde. Y un grupo caracterizado por un bajo consumo general.

# Visualizando la calidad de la Clusterización con PCA

Para poder ver qué tan separados están nuestros clústeres de datos, podemos reducir la dimensionalidad a 3 dimensiones y hacer un gráfico tridimensional. La reducción a 3 dimensiones la podemos hacer con PCA

In [ ]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go
# calculando la varianza acomulada explicada en función del número de componentes principales
pca = PCA()
pca.fit(X)

# graficando
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(pca.explained_variance_ratio_)+1)),
    y=pca.explained_variance_ratio_.cumsum(),
    mode='lines+markers',
    marker=dict(size=8),
    line=dict(width=2)
))
fig.update_layout(
    title='Varianza Acumulada Explicada por los Componentes Principales',
    xaxis_title='Número de Componentes Principales',
    yaxis_title='Varianza Acumulada Explicada',
    template='plotly_white'
)
fig.show()

Como se ve, con 3 variables principales explicamos el 52.8% de la varianza de nuestros datos.

In [ ]:
# transformando los datos originales a las componentes principales
pca = PCA(n_components=3)
pca_data = pca.fit_transform(X)


fig = go.Figure(data=[go.Scatter3d(
    x=pca_data[:,0],
    y=pca_data[:,1],
    z=pca_data[:,2],
    mode='markers',
    marker=dict(
        size=5,
        color=kmeanslabels, # color por cluster
        colorscale='Viridis',
        opacity=0.8,
        colorbar=dict(
            title='Cluster'  # Título de la leyenda
        )
    ),
    showlegend=False,  # No aplica para Scatter3d, pero puedes dejarlo
)])

fig.update_layout(
    title='Visualización 3D de los clusters, en el espacio de componentes principales (Plotly)',
    scene=dict(
        xaxis_title="pc1",
        yaxis_title="pc2",
        zaxis_title="pc3"
    ),
    width=800,
    height=800,
)

fig.show()

# Validación de los resultados con t-SNE

Otra forma de validar los resultados del algoritmo de agrupación es utilizar una forma de reducción de dimensionalidad y trazar los puntos en un plano 2D. A continuación, podemos colorearlos según el clúster al que pertenecen.

Un algoritmo popular para este propósito se llama [t-SNE](https://en.wikipedia.org/wiki/T-distributed_stochastic_neighbor_embedding). El funcionamiento interno del algoritmo está fuera del alcance de este caso, pero se puede encontrar una muy buena explicación [aquí](https://distill.pub/2016/misread-tsne/).

Lo que hay que tener en cuenta es que t-SNE no sabe nada de los clusters encontrados por K-means.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.colors

tsne = TSNE()
results_tsne = tsne.fit_transform(X)

cmap = matplotlib.colors.LinearSegmentedColormap.from_list(cluster_values, color_list)

plt.scatter(results_tsne[:,0], results_tsne[:,1],
    c=kmeanslabels,
    cmap=cmap,
    alpha=0.6,
    )

En el gráfico anterior, cada punto representa un perfil de carga diario. Se redujeron de 24 a 2 dimensiones. Teóricamente, la distancia entre los puntos en el espacio de mayor dimensión se conservó, por lo que los puntos que están cerca se refieren a perfiles de carga similares. El hecho de que la mayoría de los puntos azules, rojos y verdes estén próximos entre sí indica que la agrupación ha funcionado bien.

## Conclusiones

En este caso se ha presentado una forma de encontrar clusters del consumo de electricidad con el algoritmo K-means, y el algoritmo de clústering jerárquico. Hemos usado varias técnicas para encontrar el número óptimo de clusters, y hemos usado PCA y el t-SNE para validar los resultados.

Es importante tener en cuenta que hemos explorado sólo algunos algoritmos de clusterización entre muchos existentes. El mejor algoritmo en cada caso depende de la naturaleza de los datos y del objetivo de la clusterización.

<img src="fig/clustering_algorithms.png" alt="Descripción de la Algoritmos de Clusterización" width="800"/>